# L100 · Evaluate an Agent with MLflow Judges

You built AI calls in `00_sql_ai_functions`. Now you learn to measure quality. This is the first stop on the LLMOps spine that deepens through L200 and L300.

This notebook is self contained. It defines a tiny agent (one call to a Foundation Model), traces it with MLflow, and scores its answers with built in LLM judges over a small evaluation set of AkzoNobel coatings questions. No custom agent needs to be deployed first. The notebook does need access to the Foundation Model endpoint named in the widget.

### What you will learn
- How to trace an agent so every call is recorded.
- How to run `mlflow.genai.evaluate` with predefined LLM judges (Correctness, RelevanceToQuery, Safety) and a custom Guidelines judge.
- How to read the per question scores and the aggregate.

> Note: all data and questions are synthetic, for the workshop.


## 0. Install MLflow and set up

MLflow GenAI evaluation needs MLflow 3.1 or newer. Install it, then restart Python so the new version loads.


In [ ]:
%pip install -qq --upgrade "mlflow[databricks]>=3.1.0"
dbutils.library.restartPython()

In [ ]:
import re
import mlflow
import mlflow.deployments

# Widgets keep the notebook portable. The endpoint is the model the agent calls.
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "LLM endpoint")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", llm_endpoint):
    raise ValueError(f"Unsafe endpoint name: {llm_endpoint!r}.")

# Give each user their own MLflow experiment so runs do not collide in a shared workshop.
current_user = spark.sql("SELECT current_user()").first()[0]
experiment_path = f"/Users/{current_user}/akzo-workshop-l100-eval"
mlflow.set_experiment(experiment_path)
print(f"Agent model endpoint: {llm_endpoint}")
print(f"MLflow experiment:    {experiment_path}")

## 1. Define a tiny traced agent

The agent is one function: it takes a question and asks the Foundation Model. The `@mlflow.trace` decorator records every call, including inputs and outputs, so you can inspect and score them later. A short system prompt grounds the agent as an AkzoNobel coatings assistant.


In [ ]:
client = mlflow.deployments.get_deploy_client("databricks")

SYSTEM_PROMPT = (
    "You are an assistant for AkzoNobel, a paints and coatings company. "
    "Answer concisely and factually. If you are unsure, say so."
)

@mlflow.trace
def coatings_agent(question: str) -> str:
    """Single-turn agent: ask the Foundation Model one question."""
    response = client.predict(
        endpoint=llm_endpoint,
        inputs={
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            "max_tokens": 256,
            "temperature": 0.0,
        },
    )
    return response["choices"][0]["message"]["content"]

# Smoke test by calling the model directly, so this probe is not recorded as an
# evaluation trace beside the scored rows.
_probe = client.predict(
    endpoint=llm_endpoint,
    inputs={"messages": [{"role": "user", "content": "What does OTIF stand for in supply chain?"}], "max_tokens": 64},
)
print(_probe["choices"][0]["message"]["content"])

## 2. Build a small evaluation set

Each row has an `inputs` dictionary (the keyword arguments passed to the agent) and `expectations` (what a correct answer should contain). The `expected_facts` list is used by the Correctness judge: the answer is correct if it covers those facts, regardless of wording.


In [ ]:
eval_data = [
    {
        "inputs": {"question": "What does OTIF stand for in supply chain, and what does it measure?"},
        "expectations": {"expected_facts": ["On Time", "In Full", "delivery performance"]},
    },
    {
        "inputs": {"question": "In simple terms, what is gross margin?"},
        "expectations": {"expected_facts": ["revenue minus cost of goods sold", "expressed as a percentage of revenue"]},
    },
    {
        "inputs": {"question": "Why would a stronger US dollar hurt the euro reported margin of a European coatings maker that buys raw materials in dollars?"},
        "expectations": {"expected_facts": ["dollar costs become more expensive in euros", "margin falls"]},
    },
    {
        "inputs": {"question": "What is titanium dioxide used for in paint?"},
        "expectations": {"expected_facts": ["white pigment", "opacity or hiding power"]},
    },
    {
        "inputs": {"question": "What does customer churn mean for a commercial account?"},
        "expectations": {"expected_facts": ["customer stops buying", "lost revenue"]},
    },
]
print(f"{len(eval_data)} evaluation rows")

## 3. Evaluate with LLM judges

`mlflow.genai.evaluate` runs the agent on every row, then scores each answer with the judges you pass:

- **Correctness** compares the answer against the `expected_facts`.
- **RelevanceToQuery** checks the answer actually addresses the question.
- **Safety** flags harmful content.
- **Guidelines** is a custom judge: here it checks the answer is concise.

Each judge returns a yes or no plus a written rationale. The judges call a Databricks hosted judge model chosen by the workspace default, so scores can vary slightly between runs. To pin scores, pass a `model` argument to each scorer. The call returns a per row table and an aggregate you can open in the MLflow experiment UI.


In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Safety, Guidelines

results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=coatings_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Safety(),
        Guidelines(name="conciseness", guidelines="The response must be concise, at most four sentences."),
    ],
)

print("Aggregate metrics:")
for k, v in results.metrics.items():
    print(f"  {k}: {v}")

## 4. Inspect the per question scores

The result table has one row per question with each judge's verdict and rationale. This is where you see *why* an answer passed or failed, which is the signal you act on when improving an agent.


In [ ]:
eval_df = results.tables["eval_results"]
display(eval_df)

## Wrap up

You traced an agent and scored it with real LLM judges, without deploying a custom agent. This is the evaluation pattern you reuse everywhere:

| Tier | What you evaluate |
| --- | --- |
| L100 (here) | A single call agent, with Correctness, Relevance, Safety, and a custom guideline |
| L200 | A tool calling agent, with a fuller judge suite and an evaluation dataset in Unity Catalog |
| L300 | The multi domain supervisor, with production monitoring that uses these same judges on live traffic |

### Next
Open `03_short_term_memory.ipynb` to give an agent memory across turns.
